# T3.3 FAIR4ML 
[Owner: C]

Produce FAIR4ML-compliant metadata for each trained ML model. Must include: reference to training
dataset (with DOI), algorithm name and version, all hyperparameters, evaluation metrics and their
values, intended use, and known limitations. Upload the FAIR4ML metadata file alongside the model in
the TUWRD deposit (T3.9) and reference it in the RO-Crate (T3.1).

*Author*: *Ambrogi Federico*

---------------------------------------------------------------------------------------------------------------


This step implements **FAIR4ML-compliant** metadata generation for the trained machine learning models. 

The objective is to ensure that each model is fully documented, reproducible, and reusable according to FAIR principles. 


A structured metadata template is automatically generated directly from the trained scikit-learn model object and exported in machine-readable JSON format.


The metadata includes:
- algorithm name
- software library and version
- model hyperparameters
- evaluation metrics
- references to the training dataset (DOI)


Information such as the intended use of the model and known limitations are provided.

To improve reproducibility and reduce manual documentation errors, 
model hyperparameters are extracted automatically using the `get_params()` method provided by scikit-learn. 

The generated FAIR4ML metadata file is stored alongside the trained models, 
and they are referenced within the RO-Crate and TUWRD deposit workflow.

In [1]:
import sys
import os

import json
import sklearn
from datetime import datetime
from pathlib import Path
import pickle

sys.path.append(os.path.abspath(".."))

from src.preprocessing import *
from src.models import *
from src.evaluation import *
from src.utils import *

from sklearn.model_selection import train_test_split
import pandas as pd

### Loading saved models

In [2]:
with open('../outputs/models/model_multi_randomforest.pkl','rb') as f:
    model_multi = pickle.load(f)

with open('../outputs/models/model_cd_randomforest.pkl','rb') as f:
    model_cd = pickle.load(f)

params = model_cd.get_params()
print('*** Parameters Cd Model::: ')
print(params)

params = model_multi.get_params()
print('*** Parameters Multi Model::: ')
print(params)

*** Parameters Cd Model::: 
{'memory': None, 'steps': [('imputer', SimpleImputer()), ('model', RandomForestRegressor(random_state=42))], 'transform_input': None, 'verbose': False, 'imputer': SimpleImputer(), 'model': RandomForestRegressor(random_state=42), 'imputer__add_indicator': False, 'imputer__copy': True, 'imputer__fill_value': None, 'imputer__keep_empty_features': False, 'imputer__missing_values': nan, 'imputer__strategy': 'mean', 'model__bootstrap': True, 'model__ccp_alpha': 0.0, 'model__criterion': 'squared_error', 'model__max_depth': None, 'model__max_features': 1.0, 'model__max_leaf_nodes': None, 'model__max_samples': None, 'model__min_impurity_decrease': 0.0, 'model__min_samples_leaf': 1, 'model__min_samples_split': 2, 'model__min_weight_fraction_leaf': 0.0, 'model__monotonic_cst': None, 'model__n_estimators': 100, 'model__n_jobs': None, 'model__oob_score': False, 'model__random_state': 42, 'model__verbose': 0, 'model__warm_start': False}
*** Parameters Multi Model::: 
{'me

In [8]:
params.keys()

dict_keys(['memory', 'steps', 'transform_input', 'verbose', 'imputer', 'model', 'imputer__add_indicator', 'imputer__copy', 'imputer__fill_value', 'imputer__keep_empty_features', 'imputer__missing_values', 'imputer__strategy', 'model__bootstrap', 'model__ccp_alpha', 'model__criterion', 'model__max_depth', 'model__max_features', 'model__max_leaf_nodes', 'model__max_samples', 'model__min_impurity_decrease', 'model__min_samples_leaf', 'model__min_samples_split', 'model__min_weight_fraction_leaf', 'model__monotonic_cst', 'model__n_estimators', 'model__n_jobs', 'model__oob_score', 'model__random_state', 'model__verbose', 'model__warm_start'])

### Define Metadata function builder FAIRML compliant

In [22]:
# Define a Wrapper function to generate the JSON files
def generate_fair4ml_metadata(
    model,
    model_data,
    dataset_data,
    output_file=None, ):
    
    """
    Generate FAIR4ML metadata in JSON format 
    from a trained ML model.
    """

    metadata = {
            # general information
            "fair4ml:dateCreated": datetime.utcnow().isoformat() + "Z",

            # model information
            "fair4ml:model": {
                "fair4ml:name" : model_data['model_name'],
                "fair4ml:task" : model_data['task'],
                "fair4ml:intendedUse": model_data['intended_use'],
                "fair4ml:mlTask": model_data['task'],
                "fair4ml:limitation": model_data['limitation'],

                
                "fair4ml:modelDetails": {
                    "schema:name": type(model).__name__,
                    "library": "scikit-learn",
                    "library_version": sklearn.__version__,
                    "fair4ml:performance": evaluation_metrics,
                    "fair4ml:mlTask": model_data['task'],
                }         
                    
            },

            # trainig data information
            "fair4ml:trainingData": {
                "schema:name": dataset_data['dataset_name'],
                "schema:author": dataset_data['author'],
                "schema:doi": dataset_data['doi'],                
            },

    }

    
    # Save JSON file
    if output_file is not None:
        output_path = Path(output_file)

        with open(output_path, "w") as f:
            json.dump(metadata, f, indent=4)

        print(f"\n\n \t *** FAIR4ML metadata saved to: {output_path} \t \t" )

    return metadata

SyntaxError: unexpected character after line continuation character (1506369238.py, line 53)

## Create FAIRML compliant JSON files for the models


In [23]:
# --- Dataset information
dataset_name = 'Concentrations of major ions in wet precipitation samples in Austria'
doi = '10.48436/b0g4h-rv840.' 
author = 'Kasper-Giebl Anne1'

dataset_data = {'dataset_name':dataset_name,
                'doi':doi,
                'author': author,                            
                }


# --- Model information
model_name = 'sklearn_randomForest'
task = 'Regression'
evaluation_metrics = 'rms,mae'
known_limitations = "Highly inaccurate, check with in-situ measurements whenever possible" + "Sparse spatial coverage." + "Heavy metal measurements contain missing values." + "No validation outside the training period." 
intended_use = 'Prediction of heavy metals in precipitations'

model_data = {
    'model_name' :  model_name,
    'intended_use' : intended_use,
    'limitation' : known_limitations,
    'metrics' : evaluation_metrics,
    'task' : task
}

In [21]:
# --- Cd Model
with open('../outputs/models/model_cd_randomforest.pkl','rb') as f:
    model_cd = pickle.load(f)

params = model_cd.get_params()
print('*** Parameters Cd Model::: ')
print(params)

generate_FAIRML = generate_fair4ml_metadata(
    model_cd,
    model_data,
    dataset_data,
    output_file='../outputs/metadata/FAIRML_model_cd'
)


# --- Multi Model
with open('../outputs/models/model_multi_randomforest.pkl','rb') as f:
    model_multi = pickle.load(f)

model_multi.get_params()
print('*** Parameters Multi Model::: ')
print(params)

generate_FAIRML = generate_fair4ml_metadata(
    model_multi,
    model_data,
    dataset_data,
    output_file='../outputs/metadata/FAIRML_model_multi'
)

*** Parameters Cd Model::: 
{'memory': None, 'steps': [('imputer', SimpleImputer()), ('model', RandomForestRegressor(random_state=42))], 'transform_input': None, 'verbose': False, 'imputer': SimpleImputer(), 'model': RandomForestRegressor(random_state=42), 'imputer__add_indicator': False, 'imputer__copy': True, 'imputer__fill_value': None, 'imputer__keep_empty_features': False, 'imputer__missing_values': nan, 'imputer__strategy': 'mean', 'model__bootstrap': True, 'model__ccp_alpha': 0.0, 'model__criterion': 'squared_error', 'model__max_depth': None, 'model__max_features': 1.0, 'model__max_leaf_nodes': None, 'model__max_samples': None, 'model__min_impurity_decrease': 0.0, 'model__min_samples_leaf': 1, 'model__min_samples_split': 2, 'model__min_weight_fraction_leaf': 0.0, 'model__monotonic_cst': None, 'model__n_estimators': 100, 'model__n_jobs': None, 'model__oob_score': False, 'model__random_state': 42, 'model__verbose': 0, 'model__warm_start': False}


 	 *** FAIR4ML metadata saved to: